# Word Count Predictions
This notebook converts each review description into a word count vector using CountVectorizer

The length of these vectors is equal to the number of unique words in the dataset

Two models, one Naive Bayes and one Neural Network, are trained on this data to predict the wine variety

## Filter Data

In [3]:
import pandas as pd

df = pd.read_csv('Wine Reviews Dataset/winemag-data-130k-v2.csv')

varieties = [
    'Pinot Noir',
    'Chardonnay',
    'Cabernet Sauvignon',
    'Red Blend',
    'Bordeaux-style Red Blend',
    'Riesling',
    'Sauvignon Blanc',
    'Syrah',
    'Rosé',
    'Merlot'
]

# Keep only the rows with the varieties listed above
df = df[df['variety'].isin(varieties)]

# Replace the wine names with [WINE] so the model doesn't just learn the names of the wines
df['description'] = df['description'].replace(varieties, '[WINE]')

# Discard columns we won't be using
df = df.drop(columns=['Unnamed: 0', 'country', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'winery'])

df[0:10]

,description,variety
3,"Pineapple rind, lemon pith and orange blossom ...",Riesling
4,"Much like the regular bottling from 2012, this...",Pinot Noir
10,"Soft, supple plum envelopes an oaky structure ...",Cabernet Sauvignon
12,"Slightly reduced, this wine offers a chalky, t...",Cabernet Sauvignon
14,Building on 150 years and six generations of w...,Chardonnay
15,Zesty orange peels and apple notes abound in t...,Riesling
20,Ripe aromas of dark berries mingle with ample ...,Red Blend
21,"A sleek mix of tart berry, stem and herb, alon...",Pinot Noir
23,This wine from the Geneseo district offers aro...,Merlot
25,Oak and earth intermingle around robust aromas...,Pinot Noir


In [8]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Encode features
# Create Vectorizer object
#            TfidfVectorizer
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 1), max_features=None, max_df=0.75, binary=False)

# Fit the CountVectorizer object to the training data
vectorizer.fit(df['description'])

# Transform the training and testing data
X = vectorizer.transform(df['description'])

## Encode labels
le = LabelEncoder()
y = le.fit_transform(df['variety'])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

## Naive Bayes

In [5]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB

# Create a MultinomialNB object
nb = MultinomialNB()
# Create a ComplementNB object
# nb = ComplementNB()

# Fit the MultinomialNB object to the training data
nb.fit(X_train, y_train)

# Print the accuracy scores
print('Training accuracy: ', nb.score(X_train, y_train))
print('Testing accuracy: ', nb.score(X_test, y_test))

Training accuracy:  0.7872127872127872
Testing accuracy:  0.7452506133894147


## Feedforward Neural Network

In [10]:
from keras.models import Sequential
from keras.layers import Dense

# Create the neural network model
model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(len(varieties), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=16, validation_data=(X_test, y_test))


Epoch 1/10


2023-11-25 22:20:02.680730: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype variant and shape [57057,3]
	 [[{{node Placeholder/_0}}]]
2023-11-25 22:20:02.681217: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype int64 and shape [57057]
	 [[{{node Placeholder/_1}}]]


3567/3567 [==============================] - ETA: 0s - loss: 0.6830 - accuracy: 0.7700

2023-11-25 22:20:45.223031: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype int64 and shape [14265]
	 [[{{node Placeholder/_1}}]]


3567/3567 [==============================] - 45s 12ms/step - loss: 0.6830 - accuracy: 0.7700 - val_loss: 0.5681 - val_accuracy: 0.8116
Epoch 2/10
3567/3567 [==============================] - 19s 5ms/step - loss: 0.3595 - accuracy: 0.8774 - val_loss: 0.5913 - val_accuracy: 0.8142
Epoch 3/10
3567/3567 [==============================] - 17s 5ms/step - loss: 0.1987 - accuracy: 0.9337 - val_loss: 0.6878 - val_accuracy: 0.8093
Epoch 4/10
3567/3567 [==============================] - 17s 5ms/step - loss: 0.0961 - accuracy: 0.9687 - val_loss: 0.8762 - val_accuracy: 0.8048
Epoch 5/10
3567/3567 [==============================] - 17s 5ms/step - loss: 0.0432 - accuracy: 0.9866 - val_loss: 1.1787 - val_accuracy: 0.7961
Epoch 6/10
3567/3567 [==============================] - 18s 5ms/step - loss: 0.0219 - accuracy: 0.9936 - val_loss: 1.3226 - val_accuracy: 0.8004
Epoch 7/10
3567/3567 [==============================] - 17s 5ms/step - loss: 0.0130 - accuracy: 0.9962 - val_loss: 1.4885 - val_accuracy: 0.